# 06 — Synthesis: The Temporal Sandwich and Falsification Registry

## 1. The Argument in Three Acts

| Act | Notebook | Claim | Evidence | What it proves |
|-----|----------|-------|----------|----------------|
| **Backward** | 03 | Energy governance is politically contestable | ZA break on ETS surplus aligns with political events | Energy sovereignty is a *policy variable*, not exogenous endowment |
| **Present** | 04 | NEP predicts reserve currency share with generational lag | USA entity-specific ARDL/DOLS, Japan boundary condition | The mechanism operates; energy → monetary power is structural |
| **Forward** | 05 | TMPI maps who holds the forward option | Scenario-invariant top-3; constraint diagnostic | The mechanism generalises beyond oil; next transition is predictable |

These three legs are not independent findings. They are one argument in three temporal registers.
The backward test is the *premise*: if energy governance were exogenous, the forward test would
be merely geological. Because it is contestable, TMPI is strategic.

**Russia threads all three.** See §5 below.

In [ ]:
import sys
sys.path.append('../src')
from models import meta_analyse_entities
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display

# Load all outputs — graceful degradation if any notebook hasn't been run
def load_if_exists(path, label):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✓ {label}: {df.shape}")
        return df
    else:
        print(f"✗ {label}: NOT FOUND — run the relevant notebook first")
        return None

carbon_za     = load_if_exists('../outputs/tables/carbon_za_results.csv',       '03_backward  carbon_za_results')
main_results  = load_if_exists('../outputs/tables/main_results.csv',             '04_present   main_results')
entity_res    = load_if_exists('../outputs/tables/entity_specific_results.csv',  '04_present   entity_specific_results')
tmpi_rank     = load_if_exists('../outputs/tables/tmpi_rankings.csv',            '05_forward   tmpi_rankings')
sensitivity   = load_if_exists('../outputs/tables/thorium_timeline_sensitivity.csv', '05_forward sensitivity')
all_entities  = load_if_exists('../outputs/tables/all_entities_reference.csv',   '05_forward   all_entities')
panel         = pd.read_csv('../data/processed/panel_model_ready.csv')

## 3. Backward Leg Summary

In [ ]:
if carbon_za is not None:
    print("=== BACKWARD TEST: Zivot-Andrews Structural Break ===")
    print(carbon_za.to_string(index=False))
    print()
    aligned = carbon_za['aligned_with_political_event'].iloc[0] if len(carbon_za) > 0 else 'N/A'
    break_yr = carbon_za['break_year'].iloc[0] if len(carbon_za) > 0 else 'N/A'
    event = carbon_za['nearest_event'].iloc[0] if len(carbon_za) > 0 else 'N/A'
    print(f"Verdict: Break at {break_yr}, aligned with political event: {aligned}")
    if event:
        print(f"Nearest documented event: {event}")
    print()
    print("Interpretation: Allocation surplus regime breaks at a political event,")
    print("not at a commodity supply shock. Energy governance is politically contestable.")
else:
    print("Run 03_backward_carbon.ipynb first.")

## 4. Present Leg Summary

In [ ]:
if main_results is not None:
    print("=== PRESENT TEST: Robustness Table ===")
    print(main_results.to_string(index=False))
    print()
    sig = main_results[main_results['Significant'] == True]
    print(f"Significant specs: {len(sig)}/{len(main_results)}")
    print("Only levels specs significant — consistent with I(1) cointegration, not spurious")
    print()

if entity_res is not None:
    print("=== ENTITY-SPECIFIC RESULTS ===")
    print(entity_res.to_string(index=False))
    print()
    # Recompute I²
    try:
        meta = meta_analyse_entities(entity_res)
        i2 = meta.get('I2', 'N/A')
        print(f"I² = {i2}%")
        if isinstance(i2, (int, float)) and i2 > 75:
            print("High I² confirms: pooled estimate is inappropriate.")
            print("Entity-specific DOLS/ECM is the correct primary result.")
    except Exception as e:
        print(f"meta_analyse_entities: {e}")

## 5. Russia: The Hinge Across All Three Legs

Russia is not a boundary condition. It is the paper's most powerful illustration because
it runs through all three temporal registers simultaneously:

| Leg | Russia's role | What it shows |
|-----|---------------|---------------|
| **Backward** | Expelled from carbon markets (SWIFT 2022, EU ETS access ended) | Carbon markets are political infrastructure, not neutral mechanisms |
| **Present** | Forced bilateral energy corridors (ruble-yuan, ruble-rupee) — by necessity, not strategy | The mechanism operates under duress; energy relationships determine corridor formation |
| **Forward** | Outside the Western capture game for the next transition; TMPI 9.3 despite nuclear capacity | Geopolitical isolation changes *activation path*, not underlying potential |

> *"Russia's 2022 expulsion exposed the mechanism the paper describes: carbon markets are
> political infrastructure, monetary corridors follow energy relationships, and the coming
> transition will create new arenas of contestation from which current incumbents may again
> attempt exclusion. Russia shows what it looks like to be on the wrong side of that
> exclusion in real time."*

In [ ]:
# Show Russia natural experiment figure
russia_fig = '../outputs/figures/russia_natural_experiment.png'
if os.path.exists(russia_fig):
    display(Image(filename=russia_fig, width=800))
else:
    print("russia_natural_experiment.png not found — run 04_present_nep.ipynb")

# Russia fragmentation data
from variables import compute_russia_fragmentation
try:
    rus_data = compute_russia_fragmentation(panel)
    if rus_data is not None:
        print("\n=== Russia post-2022 fragmentation metrics ===")
        print(rus_data.to_string(index=False))
except Exception as e:
    print(f"compute_russia_fragmentation: {e}")

## 6. Forward Leg Summary

In [ ]:
if tmpi_rank is not None:
    print("=== FORWARD TEST: TMPI Rankings ===")
    print(tmpi_rank.to_string(index=False))
    print()

if sensitivity is not None:
    print("=== SCENARIO SENSITIVITY ===")
    print(sensitivity.to_string(index=False))
    # Check ranking stability
    orders = []
    for _, row in sensitivity.iterrows():
        order = (row.get('rank_1'), row.get('rank_2'), row.get('rank_3'))
        orders.append(order)
    stable = len(set(orders)) == 1
    print(f"\nTop-3 order invariant across scenarios: {stable}")
    if stable:
        print(f"Stable ranking: {orders[0][0]} > {orders[0][1]} > {orders[0][2]}")
        print("This is the forward leg's primary falsifiable claim.")

## 7. Unified Falsification Registry

In [ ]:
registry = pd.DataFrame([
    # ── Backward ──
    {'leg': 'Backward',
     'claim': 'ZA break aligns with documented political event (±2yr)',
     'falsify_if': 'Break year >2yr from any documented political event',
     'check_date': 'At submission',
     'data_source': 'eu_ets_compliance.csv'},
    {'leg': 'Backward',
     'claim': 'Phase I EUA CV exceeds oil benchmark (CV > 0.25)',
     'falsify_if': 'EUA Phase I CV < oil CV',
     'check_date': 'At submission [requires EUA price download]',
     'data_source': 'eua_prices.csv — MANUAL DOWNLOAD from Sandbag/ICAP'},
    # ── Present ──
    {'leg': 'Present',
     'claim': 'USA ARDL bounds F-stat above I(1) critical value',
     'falsify_if': 'F-stat below I(1) upper bound (Pesaran 2001)',
     'check_date': 'At submission',
     'data_source': 'panel_model_ready.csv'},
    {'leg': 'Present',
     'claim': 'USA DOLS bootstrap CI for NEP coefficient excludes zero',
     'falsify_if': 'Wild bootstrap CI [lower, upper] contains 0',
     'check_date': 'At submission',
     'data_source': 'panel_model_ready.csv'},
    {'leg': 'Present',
     'claim': 'Japan yen reserve share below GDP-predicted share (all years)',
     'falsify_if': 'Yen share consistently meets or exceeds GDP prediction',
     'check_date': 'At submission',
     'data_source': 'panel_model_ready.csv + IMF COFER'},
    # ── Forward ──
    {'leg': 'Forward',
     'claim': 'TMPI top-3 order (USA > CAN > IND) is scenario-invariant',
     'falsify_if': 'Top-3 order changes across Low/Base/High scenarios',
     'check_date': 'At submission',
     'data_source': 'thorium_timeline_sensitivity.csv'},
    {'leg': 'Forward',
     'claim': 'INR FX turnover share rises as India nuclear capacity grows',
     'falsify_if': 'INR share flat/declining despite nuclear capacity additions by 2030',
     'check_date': 'BIS Triennial Survey 2028',
     'data_source': 'BIS 2028 [not yet available]'},
    {'leg': 'Forward',
     'claim': 'INR share approaches 3–5% FX turnover by 2031',
     'falsify_if': 'INR share <2% in BIS 2031 despite nuclear program completion',
     'check_date': 'BIS Triennial Survey 2031',
     'data_source': 'BIS 2031 [not yet available]'},
    {'leg': 'Forward',
     'claim': 'AUD reserve share maintained/increased post-AUKUS nuclear capacity (~2035)',
     'falsify_if': 'AUD share declines after AUKUS nuclear capacity comes online',
     'check_date': 'IMF COFER 2035–2040',
     'data_source': 'IMF COFER 2035+ [not yet available]'},
    {'leg': 'Forward',
     'claim': 'BRL does NOT gain reserve currency share without capital account opening',
     'falsify_if': 'BRL FX turnover share rises without Chinn-Ito KAOPEN improvement',
     'check_date': 'BIS Triennial Survey 2037',
     'data_source': 'BIS 2037 [not yet available]'},
    {'leg': 'Russia (hinge)',
     'claim': 'Ruble-yuan/rupee energy corridors persist and deepen post-2022',
     'falsify_if': 'Russia returns to dollar-settled energy trade after sanctions lifted',
     'check_date': 'IEA Russia Energy Tracker 2026 onwards',
     'data_source': 'IEA Russia tracker (annual)'},
])

os.makedirs('../outputs/tables/', exist_ok=True)
registry.to_csv('../outputs/tables/falsification_registry.csv', index=False)
print("=== UNIFIED FALSIFICATION REGISTRY ===")
print()
for leg, grp in registry.groupby('leg', sort=False):
    print(f"[{leg}]")
    for _, r in grp.iterrows():
        print(f"  Claim:      {r['claim']}")
        print(f"  Falsify if: {r['falsify_if']}")
        print(f"  Check date: {r['check_date']}")
        print()
print(f"Saved: outputs/tables/falsification_registry.csv")

## 8. All-Entities Reference Table

In [ ]:
if all_entities is not None:
    print("=== ALL ENTITIES ===")
    print(all_entities.to_string(index=False))
    print()
    print("Note: COFER panel = 6 reserve currency issuers (empirical base)")
    print("      TMPI focal cases = 5 additional countries (forward prediction)")
else:
    print("Run 05_forward_tmpi.ipynb first.")

## 9. Reproducibility Checklist

Run notebooks in order. Each depends on the previous.

- [ ] `01_data_pull.ipynb` — all raw CSVs present in `data/raw/`
- [ ] `02_data_cleaning.ipynb` — `data/processed/panel_model_ready.csv` present
- [ ] `03_backward_carbon.ipynb` — `outputs/tables/carbon_za_results.csv` present
- [ ] `04_present_nep.ipynb` — `outputs/tables/main_results.csv`, `entity_specific_results.csv` present
- [ ] `05_forward_tmpi.ipynb` — `outputs/tables/tmpi_rankings.csv`, `prediction_registry.csv` present
- [ ] `06_synthesis.ipynb` — `outputs/tables/falsification_registry.csv` written
- [ ] **[OPTIONAL]** EUA spot prices manually downloaded → `data/raw/eua_prices.csv`
  - Source: Sandbag (https://sandbag.be/carbon-price-viewer/) or ICAP
  - Required for CV comparison in `03_backward_carbon.ipynb §4`
  - ZA structural break (§3) runs without it